# Train SINDy Autoencoder - Lorenz

PyTorch port of the original `train_lorenz.ipynb`. Same architecture, SINDy library and loss weights as the paper - see `../../README.md` for what changed (GPU-resident training data, `torch.func.jvp`-based derivatives) and why.

## Colab setup
Upload this repo somewhere Colab can see it, point `REPO_ROOT` at it below, and make sure **Runtime > Change runtime type > GPU** is selected.

In [ ]:
import os, sys

# Path to the SindyAutoencoders_Torch folder (this repo). If you uploaded/cloned
# it into Colab's local disk or mounted it from Google Drive, set this to that
# location - e.g. '/content/SindyAutoencoders_Torch' or
# '/content/drive/MyDrive/SindyAutoencoders_Torch'.
REPO_ROOT = '/content/SindyAutoencoders_Torch'

if not os.path.isdir(REPO_ROOT):
    # fall back to running locally, from within this notebook's own example folder
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

EXAMPLE_DIR = os.path.join(REPO_ROOT, 'examples', 'lorenz')
os.chdir(EXAMPLE_DIR)
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))
sys.path.insert(0, EXAMPLE_DIR)
print('Working directory:', os.getcwd())


In [ ]:
import datetime

import numpy as np
import pandas as pd

from data import get_lorenz_data
from sindy_utils import library_size
from training import train_network


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
if device == 'cpu':
    print('No GPU detected - in Colab, check Runtime > Change runtime type > GPU.')


In [ ]:
# generate training, validation, testing data
noise_strength = 1e-6
training_data = get_lorenz_data(1024, noise_strength=noise_strength)
validation_data = get_lorenz_data(20, noise_strength=noise_strength)


In [ ]:
params = {}

params['input_dim'] = 128
params['latent_dim'] = 3
params['model_order'] = 1
params['poly_order'] = 3
params['include_sine'] = False
params['library_dim'] = library_size(params['latent_dim'], params['poly_order'], params['include_sine'], True)

# sequential thresholding parameters
params['sequential_thresholding'] = True
params['coefficient_threshold'] = 0.1
params['threshold_frequency'] = 500
params['coefficient_mask'] = np.ones((params['library_dim'], params['latent_dim']))
params['coefficient_initialization'] = 'constant'

# loss function weighting
params['loss_weight_decoder'] = 1.0
params['loss_weight_sindy_z'] = 0.0
params['loss_weight_sindy_x'] = 1e-4
params['loss_weight_sindy_regularization'] = 1e-5

params['activation'] = 'sigmoid'
params['widths'] = [64, 32]

# training parameters
params['epoch_size'] = training_data['x'].shape[0]
params['batch_size'] = 1024
params['learning_rate'] = 1e-3
params['shuffle'] = False  # matches the original paper's fixed sequential batching

params['data_path'] = os.getcwd() + '/'
params['print_progress'] = True
params['print_frequency'] = 100

# training time cutoffs
params['max_epochs'] = 5001
params['refinement_epochs'] = 1001


In [ ]:
num_experiments = 1
df = pd.DataFrame()
for i in range(num_experiments):
    print('EXPERIMENT %d' % i)

    params['coefficient_mask'] = np.ones((params['library_dim'], params['latent_dim']))
    params['save_name'] = 'lorenz_' + datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S_%f")

    results_dict = train_network(training_data, validation_data, params, device=device)
    df = pd.concat([df, pd.DataFrame([{**results_dict, **params}])], ignore_index=True)

df.to_pickle('experiment_results_' + datetime.datetime.now().strftime("%Y%m%d%H%M") + '.pkl')
